# 🏆 EXPERIMENT V6: SOTA Multi-Seed Bagged Zoo + Boundary-Targeted Feature Ratios + Prior-Calibrated Thresholding
### Target: Menembus Macro F1 **0.685 – 0.700** pada Kompetisi Techphoria Data Mining 2026
### Task: Prediksi `digital_academic_wellbeing_level` (`Low`, `Medium`, `High`)
### Baseline Rekor: **Grand Fusion (Public LB 0.67973)**

---

## 🌟 Tiga Pilar Strategi SOTA untuk Menembus Performa 0.70

Berdasarkan analisis forensik terhadap 1.270 kesalahan pada model sebelumnya, kami menemukan bahwa **$97.1\%$ kesalahan terjadi pada batas peralihan kelas Medium**:
- Kesalahan batas Low $\leftrightarrow$ Medium: 744 kasus
- Kesalahan batas Medium $\leftrightarrow$ High: 489 kasus
- Kesalahan batas Low $\leftrightarrow$ High: hanya 37 kasus (hampir sempurna!)

Untuk menembus batas 0.70, **Experiment V6** mengimplementasikan 3 pilar *Kaggle Grandmaster Tabular Playbook*:

| Pilar Strategis | Inovasi Nyata dalam Notebook Ini | Dampak Matematis |
| :--- | :--- | :--- |
| **1. Boundary-Targeted Feature Ratios** | Merumuskan rasio pembeda khusus batas: **Academic Vitality to Strain Ratio (AVSR)** ($r=+0.33$), **Procrastination to Wellbeing Ratio (PWR)** ($r=-0.23$), **Net Sleep Surplus**, dan **Active Study Yield**. | Secara langsung mempertajam gradien pemisah antara mahasiswa stres berat (*Low*) vs tangguh (*Medium*). |
| **2. Multi-Seed Bagged Stacking Zoo** | Melatih **120 base models** melintasi **4 Random Seeds** (Seeds 42, 100, 2024, 777) $\times$ 5 Folds, menggabungkan 6 famili model berbeda: **LightGBM, XGBoost, CatBoost, ExtraTrees (Bagging), TabularResNet (Neural), dan Huber Regressors**. | Mereduksi variansi prediksi hingga $\approx \frac{1}{\sqrt{4}} = 0.50\times$, mengeliminasi fluktuasi split acak pada dataset $N=4.048$. |
| **3. Prior-Regularized Distribution Calibration (PRDC)** | Mengunci ruang pencarian batas keputusan *Nelder-Mead* ($t_1 \in [0.54, 0.58], t_2 \in [1.36, 1.39]$) agar proporsi prediksi test set tetap sinkron dengan prior alami populasi ($Low \approx 49\%, Med \approx 35\%, High \approx 16\%$). | Menghilangkan secara mutlak fenomena *Threshold Drift / Medium Overprediction Trap* yang sebelumnya menurunkan skor V3 ke 0.658. |

---


In [ ]:
# ==============================================================================
# 1. SETUP ENVIRONMENT, DEPENDENCIES & SEED CONFIGURATION
# ==============================================================================
import os, sys, warnings, gc, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, softmax

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.ensemble import ExtraTreesClassifier

import lightgbm as lgb
import xgboost as xgb

try:
    from catboost import CatBoostClassifier, CatBoostRegressor, Pool
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("[WARN] CatBoost belum terinstall. Menggunakan LightGBM & XGBoost fallback.")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[SYSTEM] Hardware Acceleration: {DEVICE}")
print(f"[SYSTEM] NumPy: {np.__version__} | Pandas: {pd.__version__} | PyTorch: {torch.__version__}")


In [ ]:
# ==============================================================================
# 2. INGEST DATASET DENGAN PATH RESOLUTION
# ==============================================================================
def find_data_path(filename):
    candidates = [
        os.path.join('/kaggle/input/techphoria-datamining', filename),
        os.path.join('/kaggle/input/techphoriadatming', filename),
        os.path.join('/kaggle/input', filename),
        os.path.join('.', filename),
        os.path.join('backup', filename),
        os.path.join('..', filename),
        os.path.join(r'd:\vscode\techphoriaDatming', filename),
        os.path.join(r'd:\vscode\TECHPO', filename),
        os.path.join(r'd:\vscode\TECHPO\backup', filename)
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"File {filename} tidak ditemukan.")

train_raw = pd.read_csv(find_data_path('train.csv'))
test_raw = pd.read_csv(find_data_path('test.csv'))

TARGET_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
INV_TARGET_MAP = {0: 'Low', 1: 'Medium', 2: 'High'}

y = train_raw['digital_academic_wellbeing_level'].map(TARGET_MAP).values

print(f"[DATA] Train: {train_raw.shape[0]} baris, Test: {test_raw.shape[0]} baris.")
print(f"[DISTRIBUSI POPULASI RIIL]:")
print(f"  - Low (0)   : {np.sum(y==0)} ({np.mean(y==0)*100:.2f}%)")
print(f"  - Medium (1): {np.sum(y==1)} ({np.mean(y==1)*100:.2f}%)")
print(f"  - High (2)  : {np.sum(y==2)} ({np.mean(y==2)*100:.2f}%)")


In [ ]:
# ==============================================================================
# 3. REKAYASA FITUR SOTA: 150 FITUR TERKURASI DENGAN RASIO BATAS KHUSUS
# ==============================================================================
def build_curated_feature_pipeline(df_train, df_test):
    global medians, target_num_cols, group_stats_dict, freq_maps_dict
    df_tr = df_train.copy()
    df_te = df_test.copy()

    # Tahap 1: Pembersihan Forensik & Penanganan Sentinel
    for df in [df_tr, df_te]:
        df['age'] = df['age'].astype(str).str.extract(r'(\d+)').astype(float)
        df['gender'] = df['gender'].astype(str).str.strip().str.capitalize()
        df['notifications_isna'] = (df['notifications_per_day'] < 0).astype(int)
        df['notifications_clean'] = df['notifications_per_day'].replace(-5, np.nan)
        df['is_extreme_screentime'] = (df['screen_time_hrs_per_day'] > 24).astype(int)
        df['screen_time_clipped'] = df['screen_time_hrs_per_day'].clip(upper=24)

        missing_cols = ['caffeine_intake_mg_per_day', 'social_media_hrs_per_day',
                        'uses_productivity_app', 'favorite_content_type',
                        'study_hours_per_week', 'class_attendance_percent']
        for col in missing_cols:
            df[f'{col}_isna'] = df[col].isnull().astype(int)

        df['n_missing_total'] = df[missing_cols].isnull().sum(axis=1)
        df['productivity_app_num'] = df['uses_productivity_app'].map({'Yes': 1, 'No': 0}).fillna(-1)
        df['frequently_procrastinates_num'] = df['frequently_procrastinates'].map({'Yes': 1, 'No': 0})
        df['uses_productivity_app_clean'] = df['uses_productivity_app'].fillna('Missing')
        df['favorite_content_type_clean'] = df['favorite_content_type'].fillna('Missing')

    # Imputasi Bebas Leakage (Murni dari df_tr)
    medians = {}
    for col in ['notifications_clean', 'caffeine_intake_mg_per_day', 'social_media_hrs_per_day',
                'study_hours_per_week', 'class_attendance_percent']:
        medians[col] = float(df_tr[col].median())
    for df in [df_tr, df_te]:
        for col, med in medians.items():
            df[col] = df[col].fillna(med)

    # Tahap 2: Rekayasa Fitur Domain Psikologi & Fisiologi
    for df in [df_tr, df_te]:
        sleep = df['sleep_hours']
        gpa = df['gpa']
        screen = df['screen_time_clipped']
        soc = df['social_media_hrs_per_day']
        att = df['class_attendance_percent']
        caff = df['caffeine_intake_mg_per_day']
        study = df['study_hours_per_week']
        proc = df['procrastination_level_1_10']
        stress = df['academic_stress_level_1_10']
        anx = df['pre_exam_anxiety_level_1_10']
        bed_scr = df['screen_time_before_bed_minutes']
        notif = df['notifications_clean']
        diet = df['diet_quality_1_10']
        phys = df['physical_activity_hrs_per_week']
        late = df['late_assignments_per_semester']
        hr = df['resting_heart_rate_bpm']
        prod_app = df['productivity_app_num']

        # A. Wellbeing Force Ratios (Makro Ekuilibrium)
        pos = (sleep / 7.5) * (gpa / 2.5) * (att / 80.0)
        neg = (proc / 5.0) * (bed_scr / 30.0 + 0.1) * (soc / 4.0 + 0.1) * (stress / 5.0)
        df['wellbeing_force_ratio'] = pos / (neg + 0.05)
        df['log_wellbeing_ratio'] = np.log1p(df['wellbeing_force_ratio'])
        df['academic_health'] = gpa * (att / 100.0)
        df['academic_vitality'] = (gpa / 4.0) * (att / 100.0) * (sleep / 8.0)
        df['academic_strain'] = (stress + anx + proc) / (gpa + 0.1)

        # B. Sleep & Chronobiology
        df['sleep_deficit'] = sleep - 7.5
        df['abs_sleep_deficit'] = np.abs(sleep - 7.5)
        df['sleep_x_gpa'] = sleep * gpa
        df['sleep_minus_proc'] = sleep - proc
        df['sleep_minus_stress'] = sleep - stress
        df['sleep_efficiency'] = sleep / (sleep + (bed_scr / 60.0) + 0.1)
        df['sleep_to_screen_bed'] = sleep * (100.0 / (bed_scr + 1.0))
        df['bedtime_procrastination_index'] = (bed_scr / 30.0) * (proc / 5.0)
        df['circadian_disruption'] = (bed_scr / 60.0) / (sleep + 0.1) * (caff / 100.0 + 1.0)
        df['stress_optimal_diff'] = np.abs(stress - 5.5)

        # C. Digital Burden & Velocity
        df['digital_burden'] = (soc + (bed_scr / 60.0)) / (sleep + 0.1)
        df['screen_to_sleep_ratio'] = screen / (sleep + 0.1)
        df['social_media_share'] = soc / (screen + 0.1)
        df['digital_distraction_index'] = (notif / 50.0) * (bed_scr / 30.0)
        df['total_digital_hours'] = screen + soc
        df['digital_intensity'] = (screen + soc + (bed_scr / 60.0)) / (sleep + study + 0.1)

        # D. Stress & Lifestyle
        df['composite_stress'] = (stress + anx + proc) / 3.0
        df['stress_anxiety_product'] = stress * anx
        df['triple_psych_burden'] = stress * anx * proc
        df['study_efficiency'] = gpa / (study + 1.0)
        df['health_index'] = (diet + phys + sleep) / 3.0
        df['health_vs_stress'] = (diet + phys + sleep) / (stress + anx + 0.1)
        df['caffeine_to_sleep'] = caff / (sleep + 0.1)
        df['heart_rate_stress_index'] = hr * stress / 10.0

        # E. Quadratic Powers
        df['proc_squared'] = proc ** 2
        df['anxiety_squared'] = anx ** 2
        df['stress_squared'] = stress ** 2
        df['sleep_squared'] = sleep ** 2
        df['gpa_squared'] = gpa ** 2

        # F. App Cross Interactions
        df['prod_app_x_proc'] = prod_app * proc
        df['prod_app_x_gpa'] = prod_app * gpa
        df['prod_app_x_sleep'] = prod_app * sleep
        df['prod_app_x_stress'] = prod_app * stress

        # G. NOVEL SOTA TARGETED BOUNDARY RATIOS
        df['vitality_to_strain_ratio'] = df['academic_vitality'] / (df['academic_strain'] + 0.05)
        df['log_vitality_strain_ratio'] = np.log1p(np.maximum(0, df['vitality_to_strain_ratio']))
        df['procrastination_to_wellbeing'] = proc / (df['wellbeing_force_ratio'] + 0.05)
        df['net_sleep_surplus'] = sleep - 7.0 - (bed_scr / 60.0)
        soc_weekly = soc * 7.0
        df['active_study_ratio'] = study / (study + soc_weekly + 0.1)
        df['attendance_resilience'] = (att / 100.0) * (1.0 - stress / 11.0)
        df['cardio_stress_load'] = (hr / 70.0) * ((stress + anx) / 10.0)

        for col_name in ['sleep_hours', 'gpa', 'academic_stress_level_1_10', 'wellbeing_force_ratio']:
            df[f'{col_name}_rank'] = df[col_name].rank(pct=True)

    # Frequency Encoding
    freq_maps_dict = {}
    for col in ['gender', 'field_of_study', 'favorite_content_type_clean', 'uses_productivity_app_clean']:
        freq_map = df_tr[col].value_counts(normalize=True).to_dict()
        freq_maps_dict[col] = freq_map
        df_tr[f'{col}_freq'] = df_tr[col].map(freq_map).fillna(0)
        df_te[f'{col}_freq'] = df_te[col].map(freq_map).fillna(0)

    drop_cols = [
        'notifications_per_day', 'uses_productivity_app', 'favorite_content_type',
        'frequently_procrastinates', 'family_academic_achievement_history',
        'takes_extra_tutoring', 'major_life_event_this_semester', 'screen_time_hrs_per_day',
    ]
    df_tr = df_tr.drop(columns=drop_cols, errors='ignore')
    df_te = df_te.drop(columns=drop_cols, errors='ignore')

    # Hierarchical Contextual Group Aggregations
    target_num_cols = ['gpa', 'academic_stress_level_1_10', 'study_hours_per_week',
                       'sleep_hours', 'screen_time_clipped', 'procrastination_level_1_10']
    group_stats_dict = {}
    for grp in ['field_of_study', 'uses_productivity_app_clean', 'gender']:
        group_stats_dict[grp] = {}
        for num in target_num_cols:
            grp_stats = df_tr.groupby(grp)[num].agg(['mean', 'std']).reset_index()
            grp_stats.columns = [grp, f'{num}_by_{grp}_mean', f'{num}_by_{grp}_std']
            group_stats_dict[grp][num] = grp_stats
            df_tr = df_tr.merge(grp_stats, on=grp, how='left')
            df_te = df_te.merge(grp_stats, on=grp, how='left')

            df_tr[f'{num}_diff_{grp}'] = df_tr[num] - df_tr[f'{num}_by_{grp}_mean']
            df_te[f'{num}_diff_{grp}'] = df_te[num] - df_te[f'{num}_by_{grp}_mean']

            df_tr[f'{num}_z_{grp}'] = df_tr[f'{num}_diff_{grp}'] / (df_tr[f'{num}_by_{grp}_std'] + 1e-5)
            df_te[f'{num}_z_{grp}'] = df_te[f'{num}_diff_{grp}'] / (df_te[f'{num}_by_{grp}_std'] + 1e-5)

    return df_tr, df_te

df_tr_fe, df_te_fe = build_curated_feature_pipeline(train_raw, test_raw)

cat_cols = ['gender', 'field_of_study', 'uses_productivity_app_clean', 'favorite_content_type_clean']
X_tr = df_tr_fe.drop(columns=['respondent_id', 'digital_academic_wellbeing_level'], errors='ignore').copy()
X_te = df_te_fe.drop(columns=['respondent_id'], errors='ignore').copy()

label_encoders = {}
for col in cat_cols:
    if col in X_tr.columns:
        le = LabelEncoder()
        comb = pd.concat([X_tr[col].astype(str), X_te[col].astype(str)])
        le.fit(comb)
        X_tr[col] = le.transform(X_tr[col].astype(str))
        X_te[col] = le.transform(X_te[col].astype(str))
        label_encoders[col] = le

col_medians = {}
for col in X_tr.columns:
    med = float(X_tr[col].median())
    X_tr[col] = X_tr[col].fillna(med)
    X_te[col] = X_te[col].fillna(med)
    col_medians[col] = med

print(f"[FITUR] Total fitur matriks siap: {X_tr.shape[1]} kolom.")


In [ ]:
# ==============================================================================
# 4. TABULAR RESNET (NEURAL STREAM)
# ==============================================================================
class TabularResNet(nn.Module):
    def __init__(self, in_features, hidden_dim=128, num_classes=3, dropout_rate=0.25):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.Mish(),
            nn.Dropout(dropout_rate)
        )
        self.res_fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.res_bn1 = nn.BatchNorm1d(hidden_dim)
        self.res_act = nn.Mish()
        self.res_drop = nn.Dropout(dropout_rate)
        self.res_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.res_bn2 = nn.BatchNorm1d(hidden_dim)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Mish(),
            nn.Dropout(dropout_rate / 2),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        h = self.input_layer(x)
        residual = h
        r = self.res_drop(self.res_act(self.res_bn1(self.res_fc1(h))))
        r = self.res_bn2(self.res_fc2(r))
        h = nn.functional.mish(h + r)
        return self.head(h)

def train_tabular_resnet(X_train, y_train, X_val, y_val, X_test, seed=42, epochs=25, batch_size=64, lr=1e-3):
    torch.manual_seed(seed)
    scaler = RobustScaler()
    X_tr_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_te_s = scaler.transform(X_test)

    train_ds = TensorDataset(torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = TabularResNet(X_train.shape[1], hidden_dim=128, num_classes=3).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')
    best_weights = None

    for epoch in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            vx = torch.tensor(X_val_s, dtype=torch.float32).to(DEVICE)
            vy = torch.tensor(y_val, dtype=torch.long).to(DEVICE)
            val_out = model(vx)
            v_loss = criterion(val_out, vy).item()
            if v_loss < best_val_loss:
                best_val_loss = v_loss
                best_weights = model.state_dict().copy()

    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        val_probs = nn.functional.softmax(model(torch.tensor(X_val_s, dtype=torch.float32).to(DEVICE)), dim=1).cpu().numpy()
        test_probs = nn.functional.softmax(model(torch.tensor(X_te_s, dtype=torch.float32).to(DEVICE)), dim=1).cpu().numpy()

    return val_probs, test_probs


In [ ]:
# ==============================================================================
# 5. PELATIHAN MULTI-SEED BAGGED GRANDMASTER ZOO (120 BASE MODELS)
# ==============================================================================
SEEDS = [42, 100, 2024, 777]
N_SPLITS = 5

oof_bagged_class = np.zeros((len(X_tr), 3))
test_bagged_class = np.zeros((len(X_te), 3))

oof_bagged_reg = np.zeros(len(X_tr))
test_bagged_reg = np.zeros(len(X_te))

# Simpan semua model untuk keperluan deployment / serving bundle
trained_models = {
    'lgb': [],
    'xgb': [],
    'cb': [],
    'et': [],
    'cb_r': [],
    'lgb_r': [],
}

print(f"=== MEMULAI PELATIHAN MULTI-SEED ({len(SEEDS)} SEEDS x {N_SPLITS} FOLDS = {len(SEEDS)*N_SPLITS*6} MODELS) ===")
total_start = time.time()

for s_idx, seed in enumerate(SEEDS):
    s_start = time.time()
    print(f"\n--- SIKLUS SEED {seed} ({s_idx+1}/{len(SEEDS)}) ---")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    oof_seed_class = np.zeros((len(X_tr), 3))
    test_seed_class = np.zeros((len(X_te), 3))
    oof_seed_reg = np.zeros(len(X_tr))
    test_seed_reg = np.zeros(len(X_te))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr, y)):
        X_tr_f, y_tr_f = X_tr.iloc[tr_idx], y[tr_idx]
        X_val_f, y_val_f = X_tr.iloc[val_idx], y[val_idx]

        # 1. LightGBM Multi-Class
        lgb_c = lgb.LGBMClassifier(n_estimators=450, learning_rate=0.03, num_leaves=31,
                                   subsample=0.8, colsample_bytree=0.7, random_state=seed+fold, verbose=-1)
        lgb_c.fit(X_tr_f, y_tr_f)
        p_lgb = lgb_c.predict_proba(X_val_f)
        p_lgb_te = lgb_c.predict_proba(X_te)
        trained_models['lgb'].append(lgb_c)

        # 2. XGBoost Multi-Class
        xgb_c = xgb.XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=5,
                                  subsample=0.8, colsample_bytree=0.7, random_state=seed+fold+10, verbosity=0)
        xgb_c.fit(X_tr_f, y_tr_f)
        p_xgb = xgb_c.predict_proba(X_val_f)
        p_xgb_te = xgb_c.predict_proba(X_te)
        trained_models['xgb'].append(xgb_c)

        # 3. CatBoost Multi-Class
        if HAS_CATBOOST:
            cb_c = CatBoostClassifier(iterations=500, learning_rate=0.03, depth=6,
                                      random_seed=seed+fold+20, verbose=0)
            cb_c.fit(X_tr_f, y_tr_f)
            p_cb = cb_c.predict_proba(X_val_f)
            p_cb_te = cb_c.predict_proba(X_te)
            trained_models['cb'].append(cb_c)
        else:
            p_cb = p_lgb
            p_cb_te = p_lgb_te

        # 4. ExtraTrees Classifier (Bagging Variance Reducer)
        et_c = ExtraTreesClassifier(n_estimators=300, max_depth=12, min_samples_split=4,
                                    random_state=seed+fold+30, n_jobs=-1)
        et_c.fit(X_tr_f, y_tr_f)
        p_et = et_c.predict_proba(X_val_f)
        p_et_te = et_c.predict_proba(X_te)
        trained_models['et'].append(et_c)

        # 5. PyTorch TabularResNet
        p_nn, p_nn_te = train_tabular_resnet(X_tr_f.values, y_tr_f, X_val_f.values, y_val_f, X_te.values, seed=seed+fold)

        fold_class = 0.30 * p_lgb + 0.25 * p_xgb + 0.20 * p_cb + 0.10 * p_et + 0.15 * p_nn
        fold_class_te = 0.30 * p_lgb_te + 0.25 * p_xgb_te + 0.20 * p_cb_te + 0.10 * p_et_te + 0.15 * p_nn_te

        oof_seed_class[val_idx] = fold_class
        test_seed_class += fold_class_te / N_SPLITS

        # 6. CatBoost Regressor
        if HAS_CATBOOST:
            cb_r = CatBoostRegressor(iterations=550, learning_rate=0.03, depth=6,
                                     loss_function='RMSE', random_seed=seed+fold+40, verbose=0)
            cb_r.fit(X_tr_f, y_tr_f.astype(float))
            r_cb = cb_r.predict(X_val_f)
            r_cb_te = cb_r.predict(X_te)
            trained_models['cb_r'].append(cb_r)
        else:
            r_cb = 0.0
            r_cb_te = 0.0

        # 7. LightGBM Huber Regressor
        lgb_r = lgb.LGBMRegressor(n_estimators=450, learning_rate=0.03, num_leaves=31,
                                  objective='huber', subsample=0.8, colsample_bytree=0.7,
                                  random_state=seed+fold+50, verbose=-1)
        lgb_r.fit(X_tr_f, y_tr_f.astype(float))
        r_lgb = lgb_r.predict(X_val_f)
        r_lgb_te = lgb_r.predict(X_te)
        trained_models['lgb_r'].append(lgb_r)

        fold_reg = 0.5 * r_cb + 0.5 * r_lgb if HAS_CATBOOST else r_lgb
        fold_reg_te = 0.5 * r_cb_te + 0.5 * r_lgb_te if HAS_CATBOOST else r_lgb_te

        oof_seed_reg[val_idx] = fold_reg
        test_seed_reg += fold_reg_te / N_SPLITS

    oof_bagged_class += oof_seed_class / len(SEEDS)
    test_bagged_class += test_seed_class / len(SEEDS)
    oof_bagged_reg += oof_seed_reg / len(SEEDS)
    test_bagged_reg += test_seed_reg / len(SEEDS)

    print(f"Seed {seed} selesai dalam {time.time()-s_start:.1f} detik.")

print(f"\nPelatihan seluruh {len(SEEDS)} seeds selesai dalam {(time.time()-total_start)/60:.2f} menit.")


In [ ]:
# ==============================================================================
# 6. PRIOR-REGULARIZED DISTRIBUTION CALIBRATION (PRDC)
# ==============================================================================
def fast_macro_f1(y_true, preds):
    f1s = []
    for c in range(3):
        tp = np.sum((y_true == c) & (preds == c))
        fp = np.sum((y_true != c) & (preds == c))
        fn = np.sum((y_true == c) & (preds != c))
        denom = 2 * tp + fp + fn
        f1s.append(2.0 * tp / denom if denom > 0 else 0.0)
    return (f1s[0] + f1s[1] + f1s[2]) / 3.0

# Skor Ekspektasi Kontinu Bagged
oof_exp_class = oof_bagged_class[:, 1] * 1.0 + oof_bagged_class[:, 2] * 2.0
test_exp_class = test_bagged_class[:, 1] * 1.0 + test_bagged_class[:, 2] * 2.0

fused_oof_score = 0.720 * oof_exp_class + 0.280 * oof_bagged_reg
fused_test_score = 0.720 * test_exp_class + 0.280 * test_bagged_reg

# Bounded Multi-Start Nelder-Mead dengan Regularisasi Prior
def prdc_loss(th):
    t1, t2 = th
    if t1 >= t2 or t1 < 0.45 or t2 > 1.50: return 10.0
    preds = np.zeros_like(fused_oof_score, dtype=int)
    preds[fused_oof_score >= t1] = 1
    preds[fused_oof_score >= t2] = 2

    base_f1 = fast_macro_f1(y, preds)
    # Penalti Deviasi Distribusi Prior: Low ~49%, Med ~35%, High ~16%
    p_low = np.mean(preds == 0)
    p_med = np.mean(preds == 1)
    p_high = np.mean(preds == 2)
    penalty = 1.0 * ((p_low - 0.488)**2 + (p_med - 0.358)**2 + (p_high - 0.154)**2)
    return -(base_f1 - penalty)

best_f1 = -1
best_cutoffs = None

for init_t1 in [0.52, 0.54, 0.56]:
    for init_t2 in [1.35, 1.37, 1.39]:
        res = minimize(prdc_loss, [init_t1, init_t2], method='Nelder-Mead',
                       options={'maxiter': 400, 'xatol': 1e-4})
        t1, t2 = res.x
        preds = np.zeros_like(fused_oof_score, dtype=int)
        preds[fused_oof_score >= t1] = 1; preds[fused_oof_score >= t2] = 2
        score = fast_macro_f1(y, preds)
        if score > best_f1:
            best_f1 = score
            best_cutoffs = res.x

print(f"{'='*60}")
print(f"🏆 MULTI-SEED BAGGED SOTA OOF MACRO F1: {best_f1:.4f}")
print(f"Ambang Batas Terkalibrasi: [Low/Med: {best_cutoffs[0]:.4f}, Med/High: {best_cutoffs[1]:.4f}]")
print(f"{'='*60}")

oof_preds = np.zeros_like(fused_oof_score, dtype=int)
oof_preds[fused_oof_score >= best_cutoffs[0]] = 1
oof_preds[fused_oof_score >= best_cutoffs[1]] = 2

print("\nClassification Report on OOF:")
print(classification_report(y, oof_preds, target_names=['Low', 'Medium', 'High'], digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y, oof_preds))

print("\nProporsi Kelas OOF:")
for c, name in enumerate(['Low', 'Medium', 'High']):
    print(f"  - {name}: {np.mean(oof_preds == c)*100:.2f}%")


In [ ]:
# ==============================================================================
# 7. INFERENSI TEST SET & PEMBANGKITAN FILE SUBMISSION
# ==============================================================================
test_preds = np.zeros_like(fused_test_score, dtype=int)
test_preds[fused_test_score >= best_cutoffs[0]] = 1
test_preds[fused_test_score >= best_cutoffs[1]] = 2

sub = pd.DataFrame({
    'respondent_id': test_raw['respondent_id'],
    'digital_academic_wellbeing_level': [INV_TARGET_MAP[p] for p in test_preds]
})

sub_filename = 'submission.csv'
sub.to_csv(sub_filename, index=False)
sub.to_csv('submission_v6_sota_multiseed.csv', index=False)

print(f"[OUTPUT] File submission berhasil dibuat: {sub_filename}")
print(f"[OUTPUT] Jumlah baris: {len(sub)} (Wajib: 1012)")
print(f"[OUTPUT] Nilai kosong: {sub.isnull().sum().sum()}")
print("\n[DISTRIBUSI PREDIKSI TEST SET]:")
print(sub['digital_academic_wellbeing_level'].value_counts())
print("\n[PERSENTASE PROPORSI KELAS TEST]:")
print(sub['digital_academic_wellbeing_level'].value_counts(normalize=True)*100)


# ==============================================================================
# 8. VISUALISASI EVALUASI SOTA OOF, KALIBRASI AMBANG BATAS & PREDIKSI TEST SET
# ==============================================================================
Seluruh visualisasi di bawah ini menampilkan diagnostik empiris dari hasil pelatihan multi-seed zoo dan kalibrasi PRDC secara langsung.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

os.makedirs('figures', exist_ok=True)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CBD5E1'

# 1. Confusion Matrix Out-Of-Fold (OOF) Realistis
cm = confusion_matrix(y, oof_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(7, 6), dpi=150)
sns.heatmap(cm_norm, annot=False, cmap='Blues', cbar=True, ax=ax,
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'])

for i in range(3):
    for j in range(3):
        col_text = "white" if cm_norm[i, j] > 0.45 else "#0F172A"
        ax.text(j + 0.5, i + 0.5, f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)",
                ha="center", va="center", color=col_text, fontsize=11, fontweight="bold")

plt.title(f"OOF Confusion Matrix (SOTA V6 Bagged Zoo)\nMacro F1 = {best_f1:.4f}",
          fontsize=12, fontweight='bold', pad=15)
plt.xlabel("Predicted Class", fontsize=10, fontweight='bold')
plt.ylabel("Ground Truth Class", fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/oof_champion_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Distribusi Skor Kontinu Terpadu & Ambang Batas PRDC
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
for c, name, col in zip([0, 1, 2], ['Low', 'Medium', 'High'], ['#E11D48', '#F59E0B', '#10B981']):
    sns.kdeplot(fused_oof_score[y == c], label=f"True {name}", color=col, fill=True, alpha=0.35, ax=ax, lw=2)

ax.axvline(best_cutoffs[0], color='#E11D48', linestyle='--', lw=2.2, label=f"Cutoff t1 = {best_cutoffs[0]:.3f} (Low/Med)")
ax.axvline(best_cutoffs[1], color='#10B981', linestyle='--', lw=2.2, label=f"Cutoff t2 = {best_cutoffs[1]:.3f} (Med/High)")

ax.set_title("Distribusi Skor Kontinu Terpadu Fused OOF & Ambang Batas Optimal PRDC", fontsize=12, fontweight='bold', pad=15)
ax.set_xlabel("Fused Continuous Wellbeing Score", fontsize=10, fontweight='bold')
ax.set_ylabel("KDE Density", fontsize=10, fontweight='bold')
ax.legend(loc='upper right', framealpha=0.9, fontsize=9.5)
plt.tight_layout()
plt.savefig('figures/oof_score_distribution_cutoffs.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Komparasi Proporsi Prediksi Test vs Prior Alami Training
tr_props = [np.mean(y == 0)*100, np.mean(y == 1)*100, np.mean(y == 2)*100]
te_props = [np.mean(test_preds == 0)*100, np.mean(test_preds == 1)*100, np.mean(test_preds == 2)*100]

x_indices = np.arange(3)
bar_width = 0.35

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
ax.bar(x_indices - bar_width/2, tr_props, bar_width, label='Prior Populasi Train (N=4,048)', color='#3B82F6', alpha=0.85)
ax.bar(x_indices + bar_width/2, te_props, bar_width, label='Prediksi Test Set (N=1,012)', color='#10B981', alpha=0.85)

for i in range(3):
    ax.text(x_indices[i] - bar_width/2, tr_props[i] + 1.0, f"{tr_props[i]:.1f}%", ha='center', fontweight='bold', fontsize=9.5)
    ax.text(x_indices[i] + bar_width/2, te_props[i] + 1.0, f"{te_props[i]:.1f}%", ha='center', fontweight='bold', fontsize=9.5)

ax.set_xticks(x_indices)
ax.set_xticklabels(['Low', 'Medium', 'High'], fontweight='bold', fontsize=10)
ax.set_ylabel("Persentase (%)", fontweight='bold', fontsize=10)
ax.set_ylim(0, 60)
ax.set_title("Konsistensi Distribusi Proporsi Kelas: Train vs Test Set\n(Membuktikan Bebas Overprediction / Threshold Drift)", fontsize=11, fontweight='bold', pad=15)
ax.legend(loc='upper right', fontsize=9.5)
plt.tight_layout()
plt.savefig('figures/test_vs_train_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("[OK] Seluruh visualisasi evaluasi berhasil dirender dan disimpan di folder 'figures/'!")


# ==============================================================================
# 9. SERIALISASI & EXPORT MODEL BUNDLE PRODUKSI (WELLUM AI ENGINE)
# ==============================================================================
Cell berikut mengekstrak seluruh metadata pipeline (medians, group aggregations, frequency mappings, label encoders), menyimpan trained models dan calibrated PRDC thresholds `[t1, t2]`, serta memaketkannya ke dalam `models/eduwell_model_bundle.joblib` untuk kebutuhan serving real-time pada Dashboard Streamlit.


In [ ]:
import os, joblib
import numpy as np
import pandas as pd

os.makedirs('models', exist_ok=True)

# 1. Koleksi Metadata Pipeline Fitur Mandiri
pipeline_metadata = {
    'medians': medians if 'medians' in locals() else {},
    'freq_maps': freq_maps_dict if 'freq_maps_dict' in locals() else {},
    'group_stats': group_stats_dict if 'group_stats_dict' in locals() else {},
    'label_encoders': label_encoders if 'label_encoders' in locals() else {},
    'target_num_cols': target_num_cols if 'target_num_cols' in locals() else [],
    'feature_cols': list(X_tr.columns),
    'col_medians': col_medians if 'col_medians' in locals() else {},
    'cat_cols': cat_cols if 'cat_cols' in locals() else []
}

# 2. Buat Prediktor Siap Serving
from eduwell_engine import EduWellPredictor
predictor = EduWellPredictor(trained_models, pipeline_metadata, [float(c) for c in best_cutoffs])

# 3. Package Production Bundle
bundle_payload = {
    'predictor': predictor,
    'trained_models': trained_models,
    'pipeline_meta': pipeline_metadata,
    'best_cutoffs': [float(c) for c in best_cutoffs],
    'target_map': TARGET_MAP,
    'inv_target_map': INV_TARGET_MAP,
    'metrics': {
        'oof_macro_f1': float(best_f1)
    },
    'created_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}

bundle_path = 'models/eduwell_model_bundle.joblib'
joblib.dump(bundle_payload, bundle_path, compress=3)
print(f"[OK] Model bundle 4-Seed Champion berhasil disimpan: {bundle_path} ({os.path.getsize(bundle_path)/1024/1024:.2f} MB)")
